<a href="https://colab.research.google.com/github/Addychauhan/customer_churn_prediction/blob/main/new_customer_churn_prediction_25_09_2026.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

#Importing the Required Libraries

In [ ]:
# ============================================================
# CUSTOMER CHURN PREDICTION
# ============================================================


# ------------------------------------------------------------
# IMPORT LIBRARIES
# ------------------------------------------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split


#Loading the Dataset

In [ ]:
# ------------------------------------------------------------
# LOAD THE DATASET
# ------------------------------------------------------------

# Read the CSV file
data = pd.read_csv("Churn_Modelling.csv")

#Basic Information

In [ ]:
data.head(5)

In [ ]:
data.tail(5)

In [ ]:
data.shape

In [ ]:
data.info()

In [ ]:
data.isnull().sum()

In [ ]:
data.duplicated().sum()

In [ ]:
data.describe()

In [ ]:
data.dtypes

In [ ]:
# ============================================================
# FIND NUMERICAL COLUMNS
# ============================================================

numerical_columns = data.select_dtypes(
    include="number"
).columns.tolist()

print("Numerical Columns:")
print(numerical_columns)

In [ ]:
# ============================================================
# FIND CATEGORICAL COLUMNS
# ============================================================

categorical_columns = data.select_dtypes(
    include="object"
).columns.tolist()

print("Categorical Columns:")
print(categorical_columns)

#Data Cleaning/Preprocessing

In [ ]:
data.dtypes

In [ ]:
data = data.drop(columns=['RowNumber', 'CustomerId', 'Surname'], errors='ignore')

In [ ]:
data

In [ ]:
# ============================================================
# CHECK TARGET VARIABLE
# ============================================================

print("Target variable distribution:")
print(data["Exited"].value_counts())

In [ ]:
print(
    data["Exited"].value_counts(normalize=True) * 100
)

#Explorartory Data Analysis

##Analysis of Numerical Features

In [ ]:
# Count of Target Column
data['Exited'].value_counts()

In [ ]:
# Visualization of Target Column
plt.figure(figsize=(8,5))
sns.countplot(x='Exited', data=data, width=0.4)
plt.title('Churn Prediction')
plt.show()

In [ ]:
plt.figure(figsize=(6, 4))

sns.countplot(
    x="Gender",
    hue="Exited",
    data=data
)

plt.title("Churn by Gender")

plt.show()

##Age Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data['Age'], bins=30,kde=True)
plt.title('Distribution of Age')
plt.xlabel("Age")
plt.ylabel('Count')
plt.show()

##Balance Distribution



In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data['Balance'], bins=30, kde=True)
plt.title("Balance Distribution")
plt.xlabel("Balance")
plt.ylabel('Count')
plt.show()

##Credit Score Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.histplot(data['CreditScore'],
             bins=30,
             kde=True
             )
plt.title('Credit Score Distribution')
plt.xlabel("Credit Score")
plt.ylabel("Count")
plt.show()

# Analysis of Categorical Features

##Geography Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Geography', data=data, width=0.4)
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))

sns.countplot(
    x="Geography",
    hue="Exited",
    data=data
)

plt.title("Churn by Geography")

plt.show()

##Gender Distribution

In [ ]:
plt.figure(figsize=(8,5))
sns.countplot(x='Gender', data=data, width=0.4)
plt.title("Gender Distribution")
plt.show()

# Bivariate Analysis

In [ ]:
fig, axarr=plt.subplots(3,2, figsize=(14,6))
sns.boxplot(x='Exited', y='CreditScore', data=data, ax=axarr[0][0])
sns.boxplot(x='Exited', y='Age', data=data, ax=axarr[0][1])

sns.boxplot(x='Exited', y='Tenure', data=data, ax=axarr[1][0])
sns.boxplot(x='Exited', y='Balance', data=data, ax=axarr[1][1])

sns.boxplot(x='Exited', y='NumOfProducts', data=data, ax=axarr[2][0])
sns.boxplot(x='Exited', y='EstimatedSalary', data=data, ax=axarr[2][1])
plt.tight_layout()
plt.show()

#Correlation Analysis

In [ ]:
plt.figure(figsize=(8,6))
sns.heatmap(data.corr(numeric_only=True),
          annot=True,
            cmap='coolwarm'
            )
plt.title("Correlation Heatmap")

In [ ]:
data.dtypes

#Separate Feature Columns
Only `X` is needed here, to check which columns are numeric vs. categorical before encoding. The target `y` isn't needed until after encoding, so it's defined once, below, right where the model pipeline actually uses it.

In [ ]:
# ============================================================
# SEPARATE FEATURE COLUMNS (for dtype inspection, before encoding)
# ============================================================

X = data.drop(
    "Exited",
    axis=1
)

print("Input Features:")
print(X.head())

# Checking Numerical Features and Categorical Features

In [ ]:
numerical_features = X.select_dtypes(
    include="number"
).columns.tolist()

categorical_features = X.select_dtypes(
    include="object"
).columns.tolist()

print("Numerical Features:")
print(numerical_features)

print("\nCategorical Features:")
print(categorical_features)

In [ ]:
X.head(5)

#Convert Categorical Variables into Numbers

In [ ]:
# ============================================================
# ONE-HOT ENCODING
# ============================================================

data = pd.get_dummies(
    data,
    columns=categorical_features,
    drop_first=True,
    dtype=int
)

print("\nShape after One-Hot Encoding:")
print(data.shape)

print("\nEncoded Columns:")
print(data.columns.tolist())

#Define Features and Target
This is the `X`/`y` that everything downstream (the split, scaling, resampling, models) actually uses.

In [ ]:
# ============================================================
# DEFINE FEATURES AND TARGET
# ============================================================

X = data.drop("Exited", axis=1)

y = data["Exited"]

print("\nFeature Shape:", X.shape)
print("Target Shape:", y.shape)

In [ ]:
# ============================================================
# TRAIN-VALIDATION-TEST SPLIT
# ============================================================

# First: 80% training + 20% testing

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X,
    y,
    test_size=0.20,
    random_state=42,
    stratify=y
)

In [ ]:
# Second split:
# 80% training data -> 80% actual training + 20% validation

X_train, X_val, y_train, y_val = train_test_split(
    X_train_full,
    y_train_full,
    test_size=0.20,
    random_state=42,
    stratify=y_train_full
)

print("\nData Split:")
print("Training:", X_train.shape)
print("Validation:", X_val.shape)
print("Testing:", X_test.shape)

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import (
    accuracy_score,
    precision_score,
    recall_score,
    f1_score,
    roc_auc_score,
    confusion_matrix,
    classification_report
)


In [ ]:
# ============================================================
# FEATURE SCALING
# ============================================================

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)

X_val_scaled = scaler.transform(X_val)

X_test_scaled = scaler.transform(X_test)

In [ ]:
# ============================================================
# HANDLE CLASS IMBALANCE
# ============================================================

from imblearn.combine import SMOTETomek

print("\nBefore SMOTETomek:")
print(y_train.value_counts())

smote_tomek = SMOTETomek(
    random_state=42
)

X_train_balanced, y_train_balanced = smote_tomek.fit_resample(
    X_train_scaled,
    y_train
)

print("\nAfter SMOTETomek:")
print(pd.Series(y_train_balanced).value_counts())

In [ ]:
# ============================================================
# CREATE MODELS
# ============================================================

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.naive_bayes import GaussianNB
from xgboost import XGBClassifier

models = {

    "Logistic Regression": LogisticRegression(
        C=1.0,
        max_iter=2000,
        class_weight="balanced",
        random_state=42
    ),

    "Decision Tree": DecisionTreeClassifier(
        max_depth=8,
        min_samples_split=10,
        min_samples_leaf=5,
        class_weight="balanced",
        random_state=42
    ),

    "Random Forest": RandomForestClassifier(
        n_estimators=400,
        max_depth=12,
        min_samples_split=5,
        min_samples_leaf=2,
        max_features="sqrt",
        class_weight="balanced",
        random_state=42,
        n_jobs=-1
    ),

    "XGBoost": XGBClassifier(
        n_estimators=400,
        max_depth=5,
        learning_rate=0.05,
        subsample=0.8,
        colsample_bytree=0.8,
        min_child_weight=3,
        gamma=0.1,
        reg_alpha=0.1,
        reg_lambda=1.0,
        eval_metric="logloss",
        random_state=42,
        n_jobs=-1
    ),

    "Gaussian Naive Bayes": GaussianNB()
}

In [ ]:
# ============================================================
# FUNCTION FOR THRESHOLD TUNING
# ============================================================

def find_best_threshold(model, X_validation, y_validation):

    probabilities = model.predict_proba(
        X_validation
    )[:, 1]

    thresholds = np.arange(
        0.10,
        0.91,
        0.01
    )

    best_threshold = 0.50
    best_f1 = 0

    for threshold in thresholds:

        predictions = (
            probabilities >= threshold
        ).astype(int)

        f1 = f1_score(
            y_validation,
            predictions,
            zero_division=0
        )

        if f1 > best_f1:

            best_f1 = f1
            best_threshold = threshold

    return best_threshold, best_f1

In [ ]:
# ============================================================
# TRAIN AND EVALUATE MODELS
# ============================================================

results = []

trained_models = {}

thresholds = {}


for name, model in models.items():

    print("\n" + "=" * 60)
    print("Training:", name)
    print("=" * 60)

    # Train model
    model.fit(
        X_train_balanced,
        y_train_balanced
    )

    # Store trained model
    trained_models[name] = model

    # --------------------------------------------------------
    # Find best threshold using VALIDATION data
    # --------------------------------------------------------

    best_threshold, validation_f1 = find_best_threshold(
        model,
        X_val_scaled,
        y_val
    )

    thresholds[name] = best_threshold

    print(
        "Best Validation Threshold:",
        round(best_threshold, 2)
    )

    print(
        "Validation F1:",
        round(validation_f1, 4)
    )

    # --------------------------------------------------------
    # Test prediction
    # --------------------------------------------------------

    test_probability = model.predict_proba(
        X_test_scaled
    )[:, 1]

    test_prediction = (
        test_probability >= best_threshold
    ).astype(int)

    # --------------------------------------------------------
    # Metrics
    # --------------------------------------------------------

    accuracy = accuracy_score(
        y_test,
        test_prediction
    )

    precision = precision_score(
        y_test,
        test_prediction,
        zero_division=0
    )

    recall = recall_score(
        y_test,
        test_prediction,
        zero_division=0
    )

    f1 = f1_score(
        y_test,
        test_prediction,
        zero_division=0
    )

    roc_auc = roc_auc_score(
        y_test,
        test_probability
    )

    results.append({

        "Model": name,

        "Threshold": round(
            best_threshold, 2
        ),

        "Accuracy": round(
            accuracy, 4
        ),

        "Precision": round(
            precision, 4
        ),

        "Recall": round(
            recall, 4
        ),

        "F1 Score": round(
            f1, 4
        ),

        "ROC-AUC": round(
            roc_auc, 4
        )
    })

In [ ]:
# ============================================================
# CREATE RESULTS DATAFRAME
# ============================================================

results_df = pd.DataFrame(results)

results_df = results_df.sort_values(
    by="F1 Score",
    ascending=False
)

print("\n")
print("=" * 80)
print("FINAL MODEL COMPARISON")
print("=" * 80)

print(
    results_df.to_string(
        index=False
    )
)

In [ ]:
# ============================================================
# SAVE RESULTS
# ============================================================

results_df.to_csv(
    "churn_model_comparison.csv",
    index=False
)

print(
    "\nModel comparison saved as "
    "'churn_model_comparison.csv'"
)

In [ ]:
# ============================================================
# SELECT BEST MODEL
# ============================================================

best_model_name = results_df.iloc[0]["Model"]

best_threshold = thresholds[
    best_model_name
]

best_model = trained_models[
    best_model_name
]

print("\nBest Model:")
print(best_model_name)

print(
    "Best Threshold:",
    best_threshold
)

In [ ]:
# ============================================================
# FINAL PREDICTIONS
# ============================================================

final_probability = best_model.predict_proba(
    X_test_scaled
)[:, 1]

final_prediction = (
    final_probability >= best_threshold
).astype(int)

In [ ]:
# ============================================================
# CLASSIFICATION REPORT
# ============================================================

print("\n")
print("=" * 60)
print("CLASSIFICATION REPORT")
print("=" * 60)

print(
    classification_report(
        y_test,
        final_prediction,
        target_names=[
            "Not Churned",
            "Churned"
        ],
        zero_division=0
    )
)

In [ ]:
# ============================================================
# CONFUSION MATRIX
# ============================================================

cm = confusion_matrix(
    y_test,
    final_prediction
)

plt.figure(figsize=(6, 5))

sns.heatmap(
    cm,
    annot=True,
    fmt="d",
    cmap="Blues"
)

plt.title(
    f"Confusion Matrix - {best_model_name}"
)

plt.xlabel("Predicted")

plt.ylabel("Actual")

plt.show()

In [ ]:
# ============================================================
# FEATURE IMPORTANCE
# ============================================================

if hasattr(best_model, "feature_importances_"):

    importance = pd.DataFrame({

        "Feature": X.columns,

        "Importance":
            best_model.feature_importances_

    })

    importance = importance.sort_values(
        by="Importance",
        ascending=False
    )

    print("\nTop Important Features:")
    print(importance.head(15))

    plt.figure(figsize=(10, 6))

    sns.barplot(
        data=importance.head(15),
        x="Importance",
        y="Feature"
    )

    plt.title(
        f"Top Features - {best_model_name}"
    )

    plt.show()

In [ ]:
# ============================================================
# SAVE MODEL + SCALER + FEATURES + THRESHOLD
# ============================================================
#
# XGBoost models are NOT reliably portable when pickled directly -- pickling embeds
# XGBoost's raw internal buffer, and that format can break across XGBoost versions or
# even get corrupted by some file transfers ("input stream corrupted" errors on load).
# XGBoost's own documentation recommends its native save_model()/load_model() instead,
# so: if the winning model is XGBoost, save it that way in a separate file; otherwise
# (Logistic Regression / Decision Tree / Random Forest / Naive Bayes) plain pickle is fine.

import pickle

model_package = {
    "model_name": best_model_name,
    "scaler": scaler,
    "feature_names": X.columns.tolist(),
    "threshold": best_threshold,
}

if hasattr(best_model, "save_model"):
    best_model.save_model("xgb_model.json")   # XGBoost's stable, version-portable format
    model_package["model_type"] = "xgboost"
else:
    model_package["model"] = best_model
    model_package["model_type"] = "sklearn"

with open("churn_model.pkl", "wb") as file:
    pickle.dump(model_package, file)

print("\n")
print("=" * 60)
print("MODEL SAVED SUCCESSFULLY")
print("=" * 60)
print("Best model:", best_model_name)
if model_package["model_type"] == "xgboost":
    print("Files: churn_model.pkl  +  xgb_model.json  (both needed -- keep them together)")
else:
    print("File: churn_model.pkl")

#Verify the Saved Files, Right Here
Before downloading anything, reload what was just saved and run one prediction with it -- inside this same Colab session. If this cell errors, the problem is in this notebook/environment; if it works here but still fails after downloading, the problem is a mismatched or corrupted download instead (redownload, and make sure both files transferred if XGBoost won).

In [ ]:
# ============================================================
# 25. VERIFY THE SAVED FILES BY RELOADING THEM FRESH
# ============================================================

import pickle as _pickle

with open("churn_model.pkl", "rb") as _f:
    _check = _pickle.load(_f)

print("Keys saved:", list(_check.keys()))
print("model_type:", _check["model_type"])

if _check["model_type"] == "xgboost":
    from xgboost import XGBClassifier as _XGBClassifier
    _reloaded_model = _XGBClassifier()
    _reloaded_model.load_model("xgb_model.json")
    print("Reloaded xgb_model.json successfully.")
else:
    _reloaded_model = _check["model"]

# One prediction, using the freshly reloaded model/scaler, to prove the round-trip works
_sample = X_test_scaled[[0]]
_proba = _reloaded_model.predict_proba(_sample)[:, 1][0]
print(f"Test prediction after reload: {_proba:.4f}  (should match the model's normal output)")
print("\nIf you see this message with no errors, churn_model.pkl "
      + ("and xgb_model.json are" if _check["model_type"] == "xgboost" else "is")
      + " good -- download them now.")

In [ ]:
# ============================================================
# 25. CHECK FINAL METRICS
# ============================================================

final_accuracy = accuracy_score(
    y_test,
    final_prediction
)

final_precision = precision_score(
    y_test,
    final_prediction,
    zero_division=0
)

final_recall = recall_score(
    y_test,
    final_prediction,
    zero_division=0
)

final_f1 = f1_score(
    y_test,
    final_prediction,
    zero_division=0
)

final_auc = roc_auc_score(
    y_test,
    final_probability
)

print("\n")
print("=" * 60)
print("BEST MODEL FINAL PERFORMANCE")
print("=" * 60)

print(
    f"Accuracy  : {final_accuracy:.4f}"
)

print(
    f"Precision : {final_precision:.4f}"
)

print(
    f"Recall    : {final_recall:.4f}"
)

print(
    f"F1 Score  : {final_f1:.4f}"
)

print(
    f"ROC-AUC   : {final_auc:.4f}"
)